<a href="https://colab.research.google.com/github/allusaatvikacs24-collab/AI_LAB/blob/main/8Puzzle_ManHattan_Missplaced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Method 1: A* Search using Misplaced Tiles Heuristic (With Search Tree Visualization)
import heapq

class PuzzleNode:
    def __init__(self, state, parent=None, move=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.state[r][c] == 0:
                    return r, c

def get_misplaced_tiles(state, goal):
    misplaced = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0 and val != goal[r][c]:
                misplaced += 1
    return misplaced

def solve_a_star_misplaced_tree(start_state, goal_state):
    start_h = get_misplaced_tiles(start_state, goal_state)
    start_node = PuzzleNode(start_state, g=0, h=start_h)

    open_list = []
    heapq.heappush(open_list, start_node)

    best_g = {}
    solutions = []
    optimal_g = None
    tree_log = []

    while open_list:
        current_node = heapq.heappop(open_list)
        state_tuple = tuple(tuple(row) for row in current_node.state)

        if optimal_g is not None and current_node.f > optimal_g:
            break

        if state_tuple in best_g and current_node.g > best_g[state_tuple]:
            tree_log.append((current_node, "[Dead End: Already Visited]"))
            continue
        best_g[state_tuple] = current_node.g

        if current_node.state == goal_state:
            if optimal_g is None:
                optimal_g = current_node.g
            if current_node.g == optimal_g:
                path = []
                curr = current_node
                while curr.parent:
                    path.append((curr.move, curr.state))
                    curr = curr.parent
                path.reverse()
                solutions.append(path)
                tree_log.append((current_node, "[GOAL STATE FOUND - OPTIMAL]"))
            else:
                tree_log.append((current_node, "[GOAL FOUND - Suboptimal]"))
            continue

        tree_log.append((current_node, "[Expanded]"))

        r, c = current_node.get_blank_pos()
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, move_name in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_state = [row[:] for row in current_node.state]
                new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]

                g_cost = current_node.g + 1
                new_state_tuple = tuple(tuple(row) for row in new_state)

                if new_state_tuple not in best_g or g_cost <= best_g[new_state_tuple]:
                    h_cost = get_misplaced_tiles(new_state, goal_state)
                    neighbor = PuzzleNode(new_state, parent=current_node, move=move_name, g=g_cost, h=h_cost)
                    heapq.heappush(open_list, neighbor)

    return solutions, tree_log

def print_flat_state(state):
    return "[" + ", ".join("".join(str(x) if x != 0 else "_" for x in row) for row in state) + "]"

def display_search_tree(tree_log, solutions):
    print("\n=================== SEARCH SPACE TREE ===================")
    optimal_states = set()
    for path in solutions:
        for _, state in path:
            optimal_states.add(tuple(tuple(row) for row in state))

    for node, status in tree_log:
        indent = "    " * node.g
        move_info = f"(Move: {node.move})" if node.move else "(Start State)"
        state_str = print_flat_state(node.state)

        is_on_optimal_path = tuple(tuple(row) for row in node.state) in optimal_states or node.parent is None

        highlight = "---> [SELECTED OPTIMAL PATH]" if is_on_optimal_path and "GOAL" in status else ""
        path_marker = "* " if is_on_optimal_path else "  "

        print(f"{indent}{path_marker}{move_info} {state_str} f={node.f} (g={node.g}, h={node.h}) {status} {highlight}")

def print_space_tree_diagram():
    diagram = r"""\n================= SPACE TREE DIAGRAM =================
                           [1 2 3]
                           [_ 4 6]  (g=0, h=3, f=3)  <-- Root (Start)
                           [7 5 8]
                             / \\
              (Move: Down)  /   \\  (Move: Right)
                           /     \\
                    [1 2 3]       [1 2 3]
                    [7 4 6]       [4 _ 6]  (g=1, h=2, f=3) [Expanded]
                    [_ 5 8]       [7 5 8]
                  (Pruned/          / | \
                  Suboptimal)      /  |  \_______
                    [g=1, h=4]    /   |          \
                     (f=5)       /    | (Move:    \ (Move:
                                /     |  Left)     \ Right)
                               /      |             \
                 (Move: Down) /       |              \
                             /    [1 2 3]          [1 2 3]
                            v     [_ 4 6]          [4 6 _]
                         [1 2 3]  [7 5 8]          [7 5 8]
                         [4 5 6]  (g=2, h=3, f=5)  (g=2, h=3, f=5)
                         [7 _ 8]  [Dead End:       [Dead End]
                     (g=2, h=1,    Already Visited]
                      f=3) [Expanded]
                           / \
             (Move: Left) /   \ (Move: Right)
                         /     \
                  [1 2 3]       [1 2 3]
                  [4 5 6]       [4 5 6]  (g=3, h=0, f=3) <-- [GOAL STATE]
                  [_ 7 8]       [7 8 _]
                (g=3, h=2,       [SELECTED OPTIMAL PATH]
                 f=5) [Dead End]"""
    print(diagram)

# Execution
start = [[1, 2, 3], [0, 4, 6], [7, 5, 8]]
goal = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

print("--- A* Search: Misplaced Tiles Heuristic ---")
solutions, tree_log = solve_a_star_misplaced_tree(start, goal)
print(f"Total Optimal Paths Found: {len(solutions)}")
for idx, path in enumerate(solutions, 1):
    print(f"\nPath {idx}:")
    for move, state in path:
        print(f"  -> {move}: {print_flat_state(state)}")
display_search_tree(tree_log, solutions)
print_space_tree_diagram()

--- A* Search: Misplaced Tiles Heuristic ---
Total Optimal Paths Found: 1

Path 1:
  -> Right: [123, 4_6, 758]
  -> Down: [123, 456, 7_8]
  -> Right: [123, 456, 78_]

=================== SEARCH SPACE TREE ===================
* (Start State) [123, _46, 758] f=3 (g=0, h=3) [Expanded] 
    * (Move: Right) [123, 4_6, 758] f=3 (g=1, h=2) [Expanded] 
        * (Move: Down) [123, 456, 7_8] f=3 (g=2, h=1) [Expanded] 
            * (Move: Right) [123, 456, 78_] f=3 (g=3, h=0) [GOAL STATE FOUND - OPTIMAL] ---> [SELECTED OPTIMAL PATH]
\n================= SPACE TREE DIAGRAM =================
                           [1 2 3]
                           [_ 4 6]  (g=0, h=3, f=3)  <-- Root (Start)
                           [7 5 8]
                             / \\
              (Move: Down)  /   \\  (Move: Right)
                           /     \\
                    [1 2 3]       [1 2 3]
                    [7 4 6]       [4 _ 6]  (g=1, h=2, f=3) [Expanded]
                    [_ 5 8]       [7 5 8]

In [20]:
# Method 2: A* Search using Manhattan Distance Heuristic (With Search Tree Visualization)
import heapq

class PuzzleNode:
    def __init__(self, state, parent=None, move=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.state[r][c] == 0:
                    return r, c

def get_manhattan_distance(state, goal):
    distance = 0
    goal_map = {}
    for r in range(3):
        for c in range(3):
            goal_map[goal[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                goal_r, goal_c = goal_map[val]
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance

def solve_a_star_manhattan_tree(start_state, goal_state):
    start_h = get_manhattan_distance(start_state, goal_state)
    start_node = PuzzleNode(start_state, g=0, h=start_h)

    open_list = []
    heapq.heappush(open_list, start_node)

    best_g = {}
    solutions = []
    optimal_g = None
    tree_log = []

    while open_list:
        current_node = heapq.heappop(open_list)
        state_tuple = tuple(tuple(row) for row in current_node.state)

        if optimal_g is not None and current_node.f > optimal_g:
            break

        if state_tuple in best_g and current_node.g > best_g[state_tuple]:
            tree_log.append((current_node, "[Dead End: Already Visited]"))
            continue
        best_g[state_tuple] = current_node.g

        if current_node.state == goal_state:
            if optimal_g is None:
                optimal_g = current_node.g
            if current_node.g == optimal_g:
                path = []
                curr = current_node
                while curr.parent:
                    path.append((curr.move, curr.state))
                    curr = curr.parent
                path.reverse()
                solutions.append(path)
                tree_log.append((current_node, "[GOAL STATE FOUND - OPTIMAL]"))
            else:
                tree_log.append((current_node, "[GOAL FOUND - Suboptimal]"))
            continue

        tree_log.append((current_node, "[Expanded]"))

        r, c = current_node.get_blank_pos()
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, move_name in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_state = [row[:] for row in current_node.state]
                new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]

                g_cost = current_node.g + 1
                new_state_tuple = tuple(tuple(row) for row in new_state)

                if new_state_tuple not in best_g or g_cost <= best_g[new_state_tuple]:
                    h_cost = get_manhattan_distance(new_state, goal_state)
                    neighbor = PuzzleNode(new_state, parent=current_node, move=move_name, g=g_cost, h=h_cost)
                    heapq.heappush(open_list, neighbor)

    return solutions, tree_log

def print_flat_state(state):
    return "[" + ", ".join("".join(str(x) if x != 0 else "_" for x in row) for row in state) + "]"

def display_search_tree(tree_log, solutions):
    print("\n=================== SEARCH SPACE TREE ===================")
    optimal_states = set()
    for path in solutions:
        for _, state in path:
            optimal_states.add(tuple(tuple(row) for row in state))

    for node, status in tree_log:
        indent = "    " * node.g
        move_info = f"(Move: {node.move})" if node.move else "(Start State)"
        state_str = print_flat_state(node.state)

        is_on_optimal_path = tuple(tuple(row) for row in node.state) in optimal_states or node.parent is None

        highlight = "---> [SELECTED OPTIMAL PATH]" if is_on_optimal_path and "GOAL" in status else ""
        path_marker = "* " if is_on_optimal_path else "  "

        print(f"{indent}{path_marker}{move_info} {state_str} f={node.f} (g={node.g}, h={node.h}) {status} {highlight}")

def print_space_tree_diagram():
    diagram = r"""\n================= SPACE TREE DIAGRAM =================
                           [1 2 3]
                           [_ 4 6]  (g=0, h=3, f=3)  <-- Root (Start)
                           [7 5 8]
                             / \\
              (Move: Down)  /   \\  (Move: Right)
                           /     \\
                    [1 2 3]       [1 2 3]
                    [7 4 6]       [4 _ 6]  (g=1, h=2, f=3) [Expanded]
                    [_ 5 8]       [7 5 8]
                  (Pruned/          / | \
                  Suboptimal)      /  |  \_______
                    [g=1, h=4]    /   |          \
                     (f=5)       /    | (Move:    \ (Move:
                                /     |  Left)     \ Right)
                               /      |             \
                 (Move: Down) /       |              \
                             /    [1 2 3]          [1 2 3]
                            v     [_ 4 6]          [4 6 _]
                         [1 2 3]  [7 5 8]          [7 5 8]
                         [4 5 6]  (g=2, h=3, f=5)  (g=2, h=3, f=5)
                         [7 _ 8]  [Dead End:       [Dead End]
                     (g=2, h=1,    Already Visited]
                      f=3) [Expanded]
                           / \
             (Move: Left) /   \ (Move: Right)
                         /     \
                  [1 2 3]       [1 2 3]
                  [4 5 6]       [4 5 6]  (g=3, h=0, f=3) <-- [GOAL STATE]
                  [_ 7 8]       [7 8 _]
                (g=3, h=2,       [SELECTED OPTIMAL PATH]
                 f=5) [Dead End]"""
    print(diagram)

# Execution
start = [[1, 2, 3], [0, 4, 6], [7, 5, 8]]
goal = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

print("--- A* Search: Manhattan Distance Heuristic ---")
solutions, tree_log = solve_a_star_manhattan_tree(start, goal)
print(f"Total Optimal Paths Found: {len(solutions)}")
for idx, path in enumerate(solutions, 1):
    print(f"\nPath {idx}:")
    for move, state in path:
        print(f"  -> {move}: {print_flat_state(state)}")
display_search_tree(tree_log, solutions)
print_space_tree_diagram()

--- A* Search: Manhattan Distance Heuristic ---
Total Optimal Paths Found: 1

Path 1:
  -> Right: [123, 4_6, 758]
  -> Down: [123, 456, 7_8]
  -> Right: [123, 456, 78_]

=================== SEARCH SPACE TREE ===================
* (Start State) [123, _46, 758] f=3 (g=0, h=3) [Expanded] 
    * (Move: Right) [123, 4_6, 758] f=3 (g=1, h=2) [Expanded] 
        * (Move: Down) [123, 456, 7_8] f=3 (g=2, h=1) [Expanded] 
            * (Move: Right) [123, 456, 78_] f=3 (g=3, h=0) [GOAL STATE FOUND - OPTIMAL] ---> [SELECTED OPTIMAL PATH]
\n================= SPACE TREE DIAGRAM =================
                           [1 2 3]
                           [_ 4 6]  (g=0, h=3, f=3)  <-- Root (Start)
                           [7 5 8]
                             / \\
              (Move: Down)  /   \\  (Move: Right)
                           /     \\
                    [1 2 3]       [1 2 3]
                    [7 4 6]       [4 _ 6]  (g=1, h=2, f=3) [Expanded]
                    [_ 5 8]       [7 5